## Feature Engineering

In [1]:
import os
import joblib
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine

In [2]:
load_dotenv()
engine = create_engine(
    f"mysql+mysqlconnector://{os.getenv('MYSQL_USER')}:{os.getenv('MYSQL_PASSWORD')}"
    f"@{os.getenv('MYSQL_HOST')}/{os.getenv('MYSQL_DATABASE')}"
)

In [3]:
query = """
SELECT d.record_date, d.train_no, d.station_code, d.station_no,
       d.delay_minutes, d.day_of_week, d.month, d.is_monsoon,
       d.is_extreme_delay, t.coverage_tier, s.station_zone
FROM delays d
JOIN trains t ON d.train_no = t.train_no
JOIN stations s ON d.station_code = s.station_code
"""
df = pd.read_sql(query, engine)
df["record_date"] = pd.to_datetime(df["record_date"])
df.shape

(174988, 11)

In [4]:
# %%
station_to_region = {
    # Mumbai / Raigad / Terminals
    "CSMT":"Mumbai", "CSTM":"Mumbai", "LTT":"Mumbai", "BDTS":"Mumbai", "MMCT":"Mumbai",
    "BVI":"Mumbai", "DR":"Mumbai", "TNA":"Mumbai", "KYN":"Mumbai", "PNVL":"Mumbai",
    "APTA":"Mumbai", "PEN":"Mumbai", "ROHA":"Mumbai", "KOL":"Mumbai", "MNI":"Mumbai",
    "VEER":"Mumbai", "INP":"Mumbai",
    
    # Ratnagiri district
    "KFD":"Ratnagiri", "VINH":"Ratnagiri", "SAPE":"Ratnagiri", "KHED":"Ratnagiri",
    "CHI":"Ratnagiri", "ANO":"Ratnagiri", "SVX":"Ratnagiri", "UKC":"Ratnagiri",
    "VID":"Ratnagiri", "SGR":"Ratnagiri", "RN":"Ratnagiri", "ADVI":"Ratnagiri",
    "NIV":"Ratnagiri", "ACRN":"Ratnagiri", "AVRD":"Ratnagiri", "RAJP":"Ratnagiri",
    "MADR":"Ratnagiri", "NAN":"Ratnagiri", "VRLI":"Ratnagiri", "SNGD":"Ratnagiri", "ADL":"Ratnagiri",
    
    # Sindhudurg (north — Kankavli side)
    "KKW":"Kankavli", "ZARP":"Kankavli", "VBW":"Kankavli", "AT":"Kankavli",
    "DWV":"Kankavli", "GNO":"Kankavli", "KLBN":"Kankavli", "BOKE":"Kankavli",
    "KMAH":"Kankavli", "KDVI":"Kankavli", "KRPN":"Kankavli",
    
    # Sindhudurg (south — Sawantwadi side, near Goa border)
    "SWV":"Sawantwadi", "SNDD":"Sawantwadi", "SRVX":"Sawantwadi", "KUDL":"Sawantwadi",
    "BLLI":"Sawantwadi",
    
    # Goa
    "THVM":"Madgaon", "KRMI":"Madgaon", "MAO":"Madgaon", "MJO":"Madgaon",
    "VEN":"Madgaon", "CTTP":"Madgaon", "PERN":"Madgaon", "CNO":"Madgaon", "LOL":"Madgaon",
    
    # Karnataka coast
    "KAWR":"Mangalore", "ANKL":"Mangalore", "GOK":"Mangalore", "KT":"Mangalore",
    "MRJN":"Mangalore", "HNA":"Mangalore", "MRDW":"Mangalore", "BTJL":"Mangalore",
    "MANK":"Mangalore", "SHMI":"Mangalore", "BYNR":"Mangalore", "KUDA":"Mangalore",
    "UD":"Mangalore", "PDD":"Mangalore", "NAND":"Mangalore", "MULK":"Mangalore",
    "SL":"Mangalore", "TOK":"Mangalore", "MAJN":"Mangalore", "MAQ":"Mangalore",
    "HAA":"Mangalore", "INJ":"Mangalore", "BIJR":"Mangalore", "BKJ":"Mangalore",
    "SEN":"Mangalore", "SUAL":"Mangalore", "CANO":"Mangalore"
}

# %%
check = pd.read_sql(
    """
    SELECT DISTINCT station_no AS distance_from_origin, station_code 
    FROM schedule 
    WHERE train_no = (SELECT MIN(train_no) FROM trains) 
    ORDER BY distance_from_origin
    """,
    engine
)
check["region"] = check["station_code"].map(station_to_region)
print("--- Geographic Sequence Check ---")
print(check.to_string())

# Verify 0 unmapped stations remain in your dataset
df["region"] = df["station_code"].map(station_to_region)
unmapped = df.loc[df["region"].isna(), "station_code"].unique()
print("\nUnmapped stations count:", len(unmapped))
if len(unmapped) > 0:
    print("Unmapped station codes:", sorted(unmapped))

--- Geographic Sequence Check ---
    distance_from_origin station_code      region
0                      1         CSMT      Mumbai
1                      2           DR      Mumbai
2                      3          TNA      Mumbai
3                      4         PNVL      Mumbai
4                      5          MNI      Mumbai
5                      6         KHED   Ratnagiri
6                      7          CHI   Ratnagiri
7                      8          SGR   Ratnagiri
8                      9           RN   Ratnagiri
9                     10         ADVI   Ratnagiri
10                    11         RAJP   Ratnagiri
11                    12          VBW    Kankavli
12                    13          KKW    Kankavli
13                    14         SNDD  Sawantwadi
14                    15         KUDL  Sawantwadi
15                    16          SWV  Sawantwadi
16                    17         PERN     Madgaon
17                    18         THVM     Madgaon
18              

In [5]:
# %%
weather_df = pd.read_csv("../data/raw/weather/konkan_rainfall_meteostat.csv")
weather_df["date"] = pd.to_datetime(weather_df["date"])

df = df.merge(
    weather_df[["date", "region", "rainfall_mm"]],
    left_on=["record_date", "region"], 
    right_on=["date", "region"], 
    how="left",
)

def rain_category(mm):
    if pd.isna(mm): return "Unknown"
    elif mm < 2.5: return "No rain"
    elif mm < 15.6: return "Light"
    elif mm < 64.5: return "Moderate"
    elif mm < 115.6: return "Heavy"
    else: return "Very heavy"

df["rain_category"] = df["rainfall_mm"].apply(rain_category)
if "date" in df.columns:
    df.drop(columns=["date"], inplace=True)

# Re-check rain_category distribution
print("\n--- Rain Category Distribution ---")
print(df["rain_category"].value_counts())



--- Rain Category Distribution ---
rain_category
No rain       89266
Light         31128
Moderate      25173
Unknown       22058
Heavy          5172
Very heavy     2191
Name: count, dtype: int64


In [6]:
for col in ["station_code", "train_no", "day_of_week", "rain_category", "station_zone"]:
    print(col, df[col].nunique())

station_code 66
train_no 45
day_of_week 7
rain_category 6
station_zone 4


In [7]:
categorical_cols = ["station_code", "train_no", "day_of_week", "rain_category", "station_zone"]
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.shape

(174988, 132)

### Train/Test Split + Baseline

In [8]:
df_encoded = df_encoded.sort_values("record_date")

split_date = df_encoded["record_date"].quantile(0.85, interpolation="nearest")
print(f"Split date: {split_date}")

train_df = df_encoded[df_encoded["record_date"] < split_date]
test_df = df_encoded[df_encoded["record_date"] >= split_date]

print(f"Train: {train_df.shape[0]} rows ({train_df['record_date'].min()} to {train_df['record_date'].max()})")
print(f"Test:  {test_df.shape[0]} rows ({test_df['record_date'].min()} to {test_df['record_date'].max()})")

Split date: 2025-12-17 00:00:00
Train: 148684 rows (2025-02-08 00:00:00 to 2025-12-16 00:00:00)
Test:  26304 rows (2025-12-17 00:00:00 to 2026-02-07 00:00:00)


In [9]:
drop_cols = ["record_date", "delay_minutes", "region", "rainfall_mm", "is_extreme_delay", "coverage_tier"]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df["delay_minutes"]
X_test = test_df.drop(columns=drop_cols)
y_test = test_df["delay_minutes"]

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline = LinearRegression()
baseline.fit(X_train, y_train)
preds = baseline.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print(f"Baseline - MAE: {mae:.2f} min, RMSE: {rmse:.2f} min, R2: {r2:.3f}")

Baseline - MAE: 35.74 min, RMSE: 63.03 min, R2: 0.053


In [11]:
dumb_preds = np.full_like(y_test, y_train.mean(), dtype=float)
dumb_mae = mean_absolute_error(y_test, dumb_preds)
print(f"Predict-the-mean MAE: {dumb_mae:.2f} min")
print(f"Linear regression improvement: {dumb_mae - mae:.2f} min")

Predict-the-mean MAE: 40.42 min
Linear regression improvement: 4.67 min


In [12]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)
meta_cols = ["train_no", "station_code", "month", "is_monsoon", "rain_category"]
train_meta = df.loc[train_df.index, meta_cols]
test_meta = df.loc[test_df.index, meta_cols]

joblib.dump(
    {
        "X_train": X_train, "y_train": y_train,
        "X_test": X_test, "y_test": y_test,
        "train_meta": train_meta, "test_meta": test_meta,
    },
    "../data/processed/model_ready_v1.pkl",
)
print("Saved.")

Saved.
